# CS639 Colab Runner

This notebook mounts Google Drive, installs dependencies, loads the default config, and runs a small Qwen smoke test.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/CS639'
CONFIG_PATH = f'{PROJECT_ROOT}/configs/default.yaml'
MODEL_PATH = f'{PROJECT_ROOT}/models/Qwen3-4B-Instruct-2507-FP8'
DATASET_ROOT = f'{PROJECT_ROOT}/dataset'
OUTPUT_ROOT = f'{PROJECT_ROOT}/outputs'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CONFIG_PATH   =', CONFIG_PATH)
print('MODEL_PATH    =', MODEL_PATH)
print('DATASET_ROOT  =', DATASET_ROOT)
print('OUTPUT_ROOT   =', OUTPUT_ROOT)

In [ ]:
%cd /content
!python -m pip install --upgrade pip
!python -m pip install -r "$PROJECT_ROOT/requirements.txt"

In [ ]:
import os
import sys
from pathlib import Path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

required_paths = [
    PROJECT_ROOT,
    CONFIG_PATH,
    MODEL_PATH,
    DATASET_ROOT,
]

for path in required_paths:
    exists = Path(path).exists()
    print(f'{path}:', 'OK' if exists else 'MISSING')

In [ ]:
from src.config import load_config

config = load_config(CONFIG_PATH)
config['model']['model_name_or_path'] = MODEL_PATH
config['datasets']['root_dir'] = DATASET_ROOT
config['experiment']['output_dir'] = OUTPUT_ROOT

config

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from src.llm.loader import build_llm

llm = build_llm(config['model'])
print('Loaded model from:', llm.model_name_or_path)
print('Runtime device:', llm.device)

In [ ]:
prompt = '''Solve the following question step by step, then provide a final answer.\n\nQuestion: If Alice has 12 apples and gives 5 away, how many apples does she have left?\nReasoning:'''

result = llm.generate(prompt, max_new_tokens=128, temperature=0.0, n=1)[0]
print(result['text'])
print('\nPrompt tokens:', result['prompt_tokens'])
print('Completion tokens:', result['completion_tokens'])
print('Latency (sec):', result['latency_sec'])

In [ ]:
from src.parsers.answers import extract_final_answer

print('Parsed final answer:', extract_final_answer(result['text']))

## Next Step

Once the smoke test works, the next implementation step is to fill in `src/data/datasets.py` and then run `scripts/run_experiment.py` on your frozen subsets.